# Qwen3.8 27B Uncensored — Kaggle + Ollama

This notebook downloads and runs the selected Hugging Face GGUF with Ollama, checks local inference, and optionally exposes an authenticated OpenAI-compatible API for a Windows PC or OCI machine.

**Selected model**
```text
hf.co/HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF:IQ4_XS
```

## Model persistence

The model is stored under `/kaggle/working/ollama-models`, not `/root`. Kaggle saves up to 20 GB of notebook output from `/kaggle/working`. This model is about 15.7 GB, so avoid putting other large files in that output directory.

- **First run:** the notebook downloads the model into `/kaggle/working/ollama-models`.
- **After the download:** use **Save Version → Quick Save** to snapshot the notebook and its current output files. Confirm `ollama-models` appears in the Output pane.
- **Later sessions:** use **Add Input** to attach the previous notebook's output files, or attach a private Kaggle Dataset created from those output files. The setup cell searches `/kaggle/input` for the saved `ollama-models` directory, copies it back to `/kaggle/working`, and starts Ollama without pulling the model again.

Do not use **Save & Run All** just to save the downloaded model: it creates a clean session and reruns the notebook. Use Quick Save for preserving the current interactive session output.

## Network and security

- The Ollama API listens only on `127.0.0.1:11434`.
- The notebook places a bearer-token authentication proxy in front of Ollama on `127.0.0.1:8787`.
- Cloudflare Quick Tunnel creates a temporary, publicly routable URL. The API requires the generated bearer token, but this is **not** a private network link. Do not send sensitive prompts, and keep the downloaded `.env` file secret.
- The URL and token change when the notebook session is restarted. Download a fresh client `.env` for each session. The `.env` is temporarily written under `/kaggle/working`; download it, then run the credential-cleanup cell before creating any later notebook version.

## 0. Kaggle notebook settings

Before running:

1. **Accelerator:** GPU T4x2.
2. **Internet:** On.
3. **Notebook visibility:** Private.
4. Start a session with the GPU accelerator enabled.

Both T4 GPUs have 15 GiB VRAM each. They can share model layers, but do not become one physical 30 GiB GPU. The selected IQ4_XS quant is approximately 15.7 GB on the model page. This notebook checks the actual writable space before downloading.

In [1]:
# 1. Verify GPU, runtime and storage

import os
import sys
import time
import json
import shutil
import pathlib
import subprocess
import requests

print("Python:", sys.version)
print("Platform:", sys.platform)

if not shutil.which("nvidia-smi"):
    raise RuntimeError("No NVIDIA GPU detected. Enable Kaggle GPU T4x2 and restart the session.")

gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv"],
    capture_output=True, text=True, check=True
)
print(gpu_info.stdout)

working = pathlib.Path("/kaggle/working")
working.mkdir(parents=True, exist_ok=True)
usage = shutil.disk_usage(working)
print(
    f"/kaggle/working: free={usage.free / 1024**3:.2f} GiB, "
    f"total={usage.total / 1024**3:.2f} GiB"
)
if pathlib.Path("/kaggle/input").exists():
    print("Attached input roots:", [p.name for p in pathlib.Path("/kaggle/input").iterdir()])

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: linux
index, name, memory.total [MiB], driver_version
0, Tesla T4, 15360 MiB, 580.178.04
1, Tesla T4, 15360 MiB, 580.178.04

/kaggle/working: free=19.50 GiB, total=19.52 GiB
Attached input roots: []


## 1. Inspect Kaggle's writable output space

`/kaggle/working` is the persistent notebook-output area. `/root` and `/kaggle/temp` are scratch storage and are not saved with the notebook output. The model must remain under `/kaggle/working` to be included in a saved notebook version.

In [2]:
# Check the output filesystem before installing or downloading anything large.

working_usage = shutil.disk_usage("/kaggle/working")
print(subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True, check=True).stdout)
print(f"Available output space: {working_usage.free / 1024**3:.2f} GiB")

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   88K   20G   1% /kaggle/working

Available output space: 19.50 GiB


## 2. Install Ollama and proxy dependencies

The install script may report that `systemd` is unavailable or that it could not detect the GPU. Kaggle containers commonly do not run `systemd`; this notebook starts `ollama serve` manually. The actual GPU placement is checked after inference.

In [3]:
subprocess.run(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq zstd"], check=True)
subprocess.run(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "requests", "fastapi", "uvicorn", "httpx"],
    check=True
)
print(subprocess.run(["ollama", "--version"], capture_output=True, text=True, check=True).stdout.strip())

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## 3. Configure persistent model storage and start Ollama

This is the key persistence fix. The model path is fixed to `/kaggle/working/ollama-models`; the notebook no longer picks `/root` just because it has more free space.

On a future session, attach the saved notebook output or private dataset first. The cell automatically searches for an attached `ollama-models` store and copies it into the writable output directory. If no saved copy is attached, the directory starts empty and the next cell downloads the model.

In [4]:
from pathlib import Path
import os
import shutil
import subprocess
import time
import requests

MODEL_ID = "hf.co/HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF:IQ4_XS"
OLLAMA_HOST = "127.0.0.1:11434"
OLLAMA_MODELS_DIR = "/kaggle/working/ollama-models"

# Optional: set this to an exact attached input path if automatic discovery does not find it.
# Example: "/kaggle/input/my-saved-notebook/ollama-models"
RESTORE_FROM_INPUT = None


def is_ollama_store(path):
    path = Path(path)
    blobs = path / "models" / "blobs"
    manifests = path / "models" / "manifests"
    return (
        blobs.is_dir()
        and manifests.is_dir()
        and any(p.is_file() and p.stat().st_size > 0 for p in blobs.iterdir())
        and any(p.is_file() for p in manifests.rglob("*"))
    )


def directory_size(path):
    total = 0
    for root, _, files in os.walk(path):
        for filename in files:
            try:
                total += (Path(root) / filename).stat().st_size
            except OSError:
                pass
    return total


model_dir = Path(OLLAMA_MODELS_DIR)
model_dir.mkdir(parents=True, exist_ok=True)

if not is_ollama_store(model_dir):
    restore_candidates = []
    if RESTORE_FROM_INPUT:
        restore_candidates.append(Path(RESTORE_FROM_INPUT))
    input_root = Path("/kaggle/input")
    if input_root.exists():
        restore_candidates.extend(input_root.rglob("ollama-models"))

    restore_source = next((p for p in restore_candidates if p.is_dir() and is_ollama_store(p)), None)
    if restore_source is not None:
        required_gib = directory_size(restore_source) / 1024**3
        free_gib = shutil.disk_usage("/kaggle/working").free / 1024**3
        if free_gib < required_gib + 1.0:
            raise RuntimeError(
                f"Not enough /kaggle/working space to restore {required_gib:.2f} GiB. "
                f"Only {free_gib:.2f} GiB is free."
            )
        print(f"Restoring model store from attached input: {restore_source}")
        shutil.copytree(restore_source, model_dir, dirs_exist_ok=True)
        print(f"Restore complete: {directory_size(model_dir) / 1024**3:.2f} GiB")
    else:
        print("No attached saved model store found. The next cell will download the model.")
else:
    print(f"Model store already exists: {directory_size(model_dir) / 1024**3:.2f} GiB")

# Check disk space only when a fresh download is needed.
if not is_ollama_store(model_dir):
    free_gib = shutil.disk_usage("/kaggle/working").free / 1024**3
    if free_gib < 18.0:
        raise RuntimeError(
            f"Only {free_gib:.2f} GiB is free in /kaggle/working. "
            "At least 18 GiB is recommended for this model and download overhead. "
            "Do not put the model under /root; it will not be saved with notebook output."
        )

ollama_env = os.environ.copy()
ollama_env.update({
    "OLLAMA_HOST": OLLAMA_HOST,
    "OLLAMA_MODELS": OLLAMA_MODELS_DIR,
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_KEEP_ALIVE": "15m",
    "OLLAMA_CONTEXT_LENGTH": "4096",
})
os.environ.update(ollama_env)


def ollama_is_up():
    try:
        response = requests.get(f"http://{OLLAMA_HOST}/api/version", timeout=2)
        return response.ok
    except requests.RequestException:
        return False


ollama_process = globals().get("ollama_process")
ollama_log = globals().get("ollama_log")

if not ollama_is_up():
    ollama_log = open("/tmp/ollama-server.log", "w")
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        env=ollama_env,
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

for attempt in range(60):
    if ollama_is_up():
        break
    if ollama_process is not None and ollama_process.poll() is not None:
        break
    time.sleep(2)
else:
    attempt = 60

if not ollama_is_up():
    log_path = Path("/tmp/ollama-server.log")
    print(log_path.read_text(errors="ignore")[-6000:] if log_path.exists() else "No Ollama log found.")
    raise RuntimeError("Ollama did not start. Inspect /tmp/ollama-server.log.")

print("Ollama is responding:", requests.get(f"http://{OLLAMA_HOST}/api/version", timeout=10).json())
print("Model ID:", MODEL_ID)
print("Persistent model directory:", OLLAMA_MODELS_DIR)

No attached saved model store found. The next cell will download the model.
Ollama is responding: {'version': '0.40.13'}
Model ID: hf.co/HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF:IQ4_XS
Persistent model directory: /kaggle/working/ollama-models


## 4. Download the selected Qwen model if it is not already present

If the model store was restored from an attached notebook output or dataset, this cell skips the download. Otherwise it pulls the selected IQ4_XS version directly into `/kaggle/working/ollama-models`.

In [7]:
# Pull only when the exact model is not already present.

model_list = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True,
    check=True
).stdout

if MODEL_ID not in model_list:
    free_gib = shutil.disk_usage("/kaggle/working").free / 1024**3

    if free_gib < 18.0:
        raise RuntimeError(
            f"Only {free_gib:.2f} GiB is free; refusing to start a large model download."
        )

    print("Model not found in the attached store. Downloading:", MODEL_ID)
    subprocess.run(["ollama", "pull", MODEL_ID], check=True)
else:
    print("Model already exists in the restored Ollama store; skipping download.")

print("\nDownloaded/available models:")
subprocess.run(["ollama", "list"], check=True)

print("\nPersistent store size:")
subprocess.run(["du", "-sh", OLLAMA_MODELS_DIR], check=True)

print(subprocess.run(
    ["df", "-h", "/kaggle/working"],
    capture_output=True,
    text=True,
    check=True
).stdout)

model_root = Path(OLLAMA_MODELS_DIR)

blobs = model_root / "blobs"
manifests = model_root / "manifests"

if not blobs.is_dir() or not manifests.is_dir():
    raise RuntimeError("Ollama model store is incomplete.")

if not any(manifests.rglob("*")):
    raise RuntimeError("Ollama manifests directory is empty.")

print("\nOllama model store verified.")
print("Blobs:", len(list(blobs.iterdir())))
print("Manifests:", len(list(manifests.rglob("*"))))

Model already exists in the restored Ollama store; skipping download.

Downloaded/available models:
NAME                                                                         ID              SIZE     MODIFIED      
hf.co/HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF:IQ4_XS    ecea6d713848    16 GB    2 minutes ago    

Persistent store size:
16G	/kaggle/working/ollama-models
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   16G  4.1G  80% /kaggle/working


Ollama model store verified.
Blobs: 4
Manifests: 4


## 5. Inspect the installed model

This checks metadata and confirms that Ollama can see the model. GPU residency is checked after the inference smoke test, because a model that has only been downloaded is not necessarily loaded into VRAM.

In [8]:
subprocess.run(["ollama", "show", MODEL_ID], check=True)
print("\nAvailable models:")
subprocess.run(["ollama", "list"], check=True)

  Model
    architecture        qwen35                
    parameters          1.86B                 
    context length      262144                
    embedding length    5120                  
    quantization        IQ4_XS                
    template source     gguf_chat_template    

  Capabilities
    tools         
    thinking      
    completion    
    vision        

  Projector
    architecture        clip       
    parameters          460.73M    
    embedding length    1152       
    dimensions          5120       


Available models:
NAME                                                                         ID              SIZE     MODIFIED      
hf.co/HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF:IQ4_XS    ecea6d713848    16 GB    3 minutes ago    


CompletedProcess(args=['ollama', 'list'], returncode=0)

## 6. Run a short local inference smoke test

This deliberately generates a short response rather than thousands of tokens. It warms the model and checks the local Ollama API. The model is kept loaded for later calls.

In [9]:
import requests
import json

local_chat_url = f"http://{OLLAMA_HOST}/api/chat"
payload = {
    "model": MODEL_ID,
    "messages": [{
        "role": "user",
        "content": "Explain how a limit order book works in three concise points."
    }],
    "stream": True,
    "keep_alive": -1,
    "options": {
        "num_ctx": 4096,
        "num_predict": 160,
        "temperature": 0.7,
    },
}

print("Loading model and generating a short response...\n")
with requests.post(local_chat_url, json=payload, stream=True, timeout=(15, 900)) as response:
    response.raise_for_status()
    final_stats = {}
    for line in response.iter_lines():
        if not line:
            continue
        item = json.loads(line)
        if "message" in item:
            print(item["message"].get("content", ""), end="", flush=True)
        if item.get("done"):
            final_stats = item

print("\n\nGeneration complete.")
print("Load duration (s):", round(final_stats.get("load_duration", 0) / 1e9, 3))
print("Prompt tokens:", final_stats.get("prompt_eval_count"))
print("Generated tokens:", final_stats.get("eval_count"))
eval_seconds = final_stats.get("eval_duration", 0) / 1e9
if eval_seconds > 0:
    print("Generation speed (tokens/s):", round(final_stats.get("eval_count", 0) / eval_seconds, 2))

Loading model and generating a short response...

- A limit order book lists all open buy and sell limit orders, sorted by price: buy orders from highest to lowest, sell orders from lowest to highest.
- Each order specifies a maximum price to buy or minimum price to sell, plus a quantity, so trades only occur when buyer and seller prices match.
- The best bid (highest buy price) and best ask (lowest sell price) form the market’s current price, and matching orders execute automatically when they cross.

Generation complete.
Load duration (s): 97.209
Prompt tokens: 65
Generated tokens: 131
Generation speed (tokens/s): 6.18


## 7. Verify GPU placement

Run this after the smoke test has completed. `ollama ps` reports whether the model is resident on GPU, CPU, or split between them; `nvidia-smi` reports actual GPU memory use.

In [10]:
print("Ollama model placement:")
subprocess.run(["ollama", "ps"], check=True)
print("\nNVIDIA GPU status:")
subprocess.run(["nvidia-smi"], check=True)

Ollama model placement:
NAME                                                                         ID              SIZE     PROCESSOR    CONTEXT    RUNNER      UNTIL   
hf.co/HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF:IQ4_XS    ecea6d713848    15 GB    100% GPU     4096       llamacpp    Forever    

NVIDIA GPU status:
Sat Oct 31 09:33:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+=======

CompletedProcess(args=['nvidia-smi'], returncode=0)

## 8. Test the local OpenAI-compatible API

Ollama's OpenAI-compatible endpoint is available locally at `http://127.0.0.1:11434/v1`. It is used for this local test only; port `11434` is never exposed through the tunnel.

In [11]:
local_base_url = f"http://{OLLAMA_HOST}/v1"
local_payload = {
    "model": MODEL_ID,
    "messages": [{"role": "user", "content": "Reply with exactly: QWEN-KAGGLE-OK"}],
    "stream": False,
    "max_tokens": 64,
}

local_response = requests.post(
    f"{local_base_url}/chat/completions",
    json=local_payload,
    timeout=(15, 300),
)
print("HTTP:", local_response.status_code)
local_response.raise_for_status()
print(local_response.json()["choices"][0]["message"]["content"])

HTTP: 200
QWEN-KAGGLE-OK


## 9. Put authentication in front of Ollama

Ollama's local API does not require authentication. Do not tunnel port `11434` directly.

This cell creates a small reverse proxy on `127.0.0.1:8787`. It requires `Authorization: Bearer <token>` for `/api/*` and `/v1/*`, preserves streamed responses, and exposes only a minimal unauthenticated `/healthz` endpoint. The random token is not printed; it is written to a client `.env` file in a later cell.

In [12]:
from pathlib import Path
import secrets
import os
import sys
import subprocess
import time
import requests

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
CLIENT_API_KEY = user_secrets.get_secret("OLLAMA_CLIENT_API_KEY")

proxy_source = r"""
import os
import secrets
import httpx
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import StreamingResponse

TOKEN = os.environ.get("OLLAMA_PROXY_TOKEN", "")
if not TOKEN:
    raise RuntimeError("OLLAMA_PROXY_TOKEN is not configured")

UPSTREAM = "http://127.0.0.1:11434"
app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
client = httpx.AsyncClient(
    timeout=httpx.Timeout(1800.0, connect=20.0, read=1800.0, write=1800.0, pool=20.0),
    trust_env=False,
)

@app.get("/healthz")
async def healthz():
    return {"ok": True, "service": "ollama-auth-proxy"}

@app.on_event("shutdown")
async def close_http_client():
    await client.aclose()

@app.api_route("/{path:path}", methods=["GET", "POST", "PUT", "PATCH", "DELETE"])
async def proxy(path: str, request: Request):
    allowed = path == "api" or path.startswith("api/") or path == "v1" or path.startswith("v1/")
    if not allowed:
        raise HTTPException(status_code=404, detail="Not found")

    auth = request.headers.get("authorization", "")
    scheme, _, supplied = auth.partition(" ")
    if scheme.lower() != "bearer" or not secrets.compare_digest(supplied, TOKEN):
        raise HTTPException(status_code=401, detail="Invalid API key")

    forward_headers = {}
    for key in ("content-type", "accept"):
        if key in request.headers:
            forward_headers[key] = request.headers[key]

    url = f"{UPSTREAM}/{path}"
    if request.url.query:
        url += f"?{request.url.query}"

    upstream_request = client.build_request(
        method=request.method,
        url=url,
        headers=forward_headers,
        content=await request.body(),
    )
    try:
        upstream_response = await client.send(upstream_request, stream=True)
    except httpx.HTTPError as exc:
        raise HTTPException(status_code=502, detail="Ollama upstream is unavailable") from exc

    excluded = {
        "connection", "keep-alive", "proxy-authenticate", "proxy-authorization",
        "te", "trailers", "transfer-encoding", "upgrade", "content-length",
    }
    response_headers = {
        key: value for key, value in upstream_response.headers.items()
        if key.lower() not in excluded
    }

    async def body_stream():
        try:
            async for chunk in upstream_response.aiter_raw():
                yield chunk
        finally:
            await upstream_response.aclose()

    return StreamingResponse(
        body_stream(),
        status_code=upstream_response.status_code,
        headers=response_headers,
    )
"""

proxy_file = Path("/kaggle/working/ollama_auth_proxy.py")
proxy_file.write_text(proxy_source)

proxy_env = os.environ.copy()
proxy_env["OLLAMA_PROXY_TOKEN"] = CLIENT_API_KEY
proxy_log = open("/tmp/ollama-proxy.log", "w")
proxy_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "ollama_auth_proxy:app", "--host", "127.0.0.1", "--port", "8787"],
    cwd="/kaggle/working",
    env=proxy_env,
    stdout=proxy_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

proxy_ready = False
for _ in range(40):
    try:
        health = requests.get("http://127.0.0.1:8787/healthz", timeout=2)
        if health.ok:
            proxy_ready = True
            break
    except requests.RequestException:
        pass
    if proxy_process.poll() is not None:
        break
    time.sleep(1)

if not proxy_ready:
    print(Path("/tmp/ollama-proxy.log").read_text(errors="ignore")[-5000:])
    raise RuntimeError("Authenticated proxy did not start.")

no_auth = requests.get("http://127.0.0.1:8787/v1/models", timeout=10)
assert no_auth.status_code == 401, f"Expected 401, got {no_auth.status_code}"

auth_headers = {"Authorization": f"Bearer {CLIENT_API_KEY}"}
auth_test = requests.get(
    "http://127.0.0.1:8787/v1/models",
    headers=auth_headers,
    timeout=30,
)
auth_test.raise_for_status()

print("Proxy ready.")
print("Unauthenticated request:", no_auth.status_code, "(expected 401)")
print("Authenticated /v1/models:", auth_test.status_code)

Proxy ready.
Unauthenticated request: 401 (expected 401)
Authenticated /v1/models: 200


## 10. Install Cloudflare Tunnel

This uses a Cloudflare Quick Tunnel. It does not require a Cloudflare account or custom domain. The URL is temporary and publicly routable; the bearer-token proxy protects API calls.

In [13]:
cloudflared_bin = "/usr/local/bin/cloudflared"
subprocess.run(
    [
        "bash", "-lc",
        "curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
    ],
    check=True,
)
subprocess.run([cloudflared_bin, "--version"], check=True)

cloudflared version 2026.10.7 (built 2026-10-30-17:05 UTC)


CompletedProcess(args=['/usr/local/bin/cloudflared', '--version'], returncode=0)

## 11. Start the Quick Tunnel and capture its temporary URL

In [14]:
import os
import re
import time
import pathlib
import subprocess

cloudflared_home = "/tmp/cloudflared-home"
pathlib.Path(cloudflared_home).mkdir(parents=True, exist_ok=True)
cloudflared_env = os.environ.copy()
cloudflared_env["HOME"] = cloudflared_home

cloudflared_log_path = "/tmp/cloudflared.log"
cloudflared_log = open(cloudflared_log_path, "w")
cloudflared_process = subprocess.Popen(
    [
        "/usr/local/bin/cloudflared", "tunnel", "--no-autoupdate",
        "--url", "http://127.0.0.1:8787", "--loglevel", "info",
    ],
    env=cloudflared_env,
    stdout=cloudflared_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

PUBLIC_URL = None
for _ in range(90):
    log_text = pathlib.Path(cloudflared_log_path).read_text(errors="ignore")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        PUBLIC_URL = match.group(0)
        break
    if cloudflared_process.poll() is not None:
        break
    time.sleep(1)

if not PUBLIC_URL:
    print(pathlib.Path(cloudflared_log_path).read_text(errors="ignore")[-6000:])
    raise RuntimeError("Cloudflare Quick Tunnel URL was not detected.")

print("Temporary API origin:", PUBLIC_URL)
print("OpenAI-compatible base URL:", PUBLIC_URL + "/v1")
print("The API key is not displayed. Download the client .env in the next cell.")

Temporary API origin: https://injection-dam-local-corpus.trycloudflare.com
OpenAI-compatible base URL: https://injection-dam-local-corpus.trycloudflare.com/v1
The API key is not displayed. Download the client .env in the next cell.


In [18]:

from pathlib import Path
from IPython.display import display, FileLink

client_env_path = Path("/kaggle/working/ollama-client.env")
client_env_path.parent.mkdir(parents=True, exist_ok=True)

client_env_path.write_text(
    f"OLLAMA_BASE_URL={PUBLIC_URL}/v1\n"
    f"OLLAMA_API_KEY={CLIENT_API_KEY}\n"
    f"OLLAMA_MODEL={MODEL_ID}\n"
)

client_script = r"""import argparse
import os
import sys
from dotenv import load_dotenv
from openai import OpenAI


def get_field(obj, *names):
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    elif not isinstance(obj, dict):
        obj = vars(obj) if hasattr(obj, "__dict__") else {}

    for name in names:
        value = obj.get(name) if isinstance(obj, dict) else None
        if value is not None:
            return value
    return None


def main():
    parser = argparse.ArgumentParser(
        description="Send a prompt to the Kaggle-hosted Ollama model."
    )
    parser.add_argument("prompt", nargs="+", help="Prompt text")
    parser.add_argument("--system", default=None, help="Optional system message")
    parser.add_argument("--temperature", type=float, default=0.3)
    parser.add_argument("--max-tokens", type=int, default=8192)
    parser.add_argument("--stream", action="store_true", help="Stream output")
    parser.add_argument(
        "--think",
        action="store_true",
        help="Enable model thinking; disabled by default to reserve tokens for the answer",
    )
    args = parser.parse_args()

    load_dotenv()
    base_url = os.getenv("OLLAMA_BASE_URL")
    api_key = os.getenv("OLLAMA_API_KEY")
    model = os.getenv("OLLAMA_MODEL")

    missing = [
        name for name, value in {
            "OLLAMA_BASE_URL": base_url,
            "OLLAMA_API_KEY": api_key,
            "OLLAMA_MODEL": model,
        }.items() if not value
    ]
    if missing:
        raise SystemExit("Missing .env values: " + ", ".join(missing))

    messages = []
    if args.system:
        messages.append({"role": "system", "content": args.system})
    messages.append({"role": "user", "content": " ".join(args.prompt)})

    client = OpenAI(
        base_url=base_url,
        api_key=api_key,
        timeout=1800.0,
        max_retries=1,
    )

    print(f"Requesting model: {model}", file=sys.stderr)
    print(
        f"Streaming: {args.stream} | Thinking: {args.think} | "
        f"Max output tokens: {args.max_tokens}",
        file=sys.stderr,
    )

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=args.temperature,
        max_tokens=args.max_tokens,
        stream=args.stream,
        extra_body={
            "reasoning_effort": "none" if not args.think else "high"
        },
    )

    if args.stream:
        content_seen = False
        reasoning_seen = False
        finish_reason = None

        for chunk in response:
            if not chunk.choices:
                continue

            choice = chunk.choices[0]
            finish_reason = choice.finish_reason or finish_reason
            delta = choice.delta
            content = get_field(delta, "content")
            reasoning = get_field(delta, "reasoning", "reasoning_content")

            if content:
                print(content, end="", flush=True)
                content_seen = True
            if reasoning:
                reasoning_seen = True

        print()
    else:
        if not response.choices:
            raise RuntimeError("The API returned no completion choices.")

        choice = response.choices[0]
        message = choice.message
        content = get_field(message, "content")
        reasoning = get_field(message, "reasoning", "reasoning_content")
        finish_reason = choice.finish_reason
        content_seen = bool(content)
        reasoning_seen = bool(reasoning)

        if content:
            print(content)

    if not content_seen:
        print("No final answer content was returned.", file=sys.stderr)
        print(f"Finish reason: {finish_reason or 'not provided'}", file=sys.stderr)

        if reasoning_seen:
            print(
                "The model emitted reasoning but no final answer. "
                "Try a larger --max-tokens value or keep thinking disabled.",
                file=sys.stderr,
            )
        else:
            print(
                "The response contained neither visible content nor recognized "
                "reasoning fields. Check the proxy/model response format.",
                file=sys.stderr,
            )


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nInterrupted.", file=sys.stderr)
        raise SystemExit(130)
    except Exception as exc:
        print(f"\nRequest failed: {type(exc).__name__}: {exc}", file=sys.stderr)
        raise SystemExit(1)
"""

client_script_path = Path("/kaggle/working/gpu_client.py")
client_script_path.write_text(client_script)

requirements_path = Path("/kaggle/working/requirements-client.txt")
requirements_path.write_text(
    "openai>=1.30.0\n"
    "python-dotenv>=1.0.1\n"
)

print("Client files generated successfully.")
print("Download these files:")

display(
    FileLink(
        str(client_env_path),
        result_html_prefix="1. Download private client .env: "
    )
)
display(
    FileLink(
        str(client_script_path),
        result_html_prefix="2. Download updated client script: "
    )
)
display(
    FileLink(
        str(requirements_path),
        result_html_prefix="3. Download client requirements: "
    )
)

print("Keep ollama-client.env secret.")
print("Rename it to .env beside gpu_client.py.")

Client files generated successfully.
Download these files:


/kaggle/working/ollama-client.env

/kaggle/working/gpu_client.py

/kaggle/working/requirements-client.txt

Keep ollama-client.env secret.
Rename it to .env beside gpu_client.py.


## 12. Create downloadable client files for Windows and OCI

The API key is stored in a temporary file under `/kaggle/temp` so it is not included in the notebook's saved output files. The client script and requirements file contain no secrets and are written to `/kaggle/working` for download.

### Remove the downloaded credential from notebook output

After downloading `ollama-client.env`, run the next cell. It deletes the token file from `/kaggle/working` so a future notebook version does not store the API secret. Keep your downloaded local copy private.


In [19]:
from pathlib import Path
secret_file = Path("/kaggle/working/ollama-client.env")
if secret_file.exists():
    secret_file.unlink()
    print("Client credential file removed from Kaggle output.")
else:
    print("No temporary client credential file found.")


Client credential file removed from Kaggle output.


## 13. Test the public endpoint through Cloudflare

The first request must be rejected with HTTP 401. The bearer-authenticated request should return the model list.

In [20]:
public_headers = {"Authorization": f"Bearer {CLIENT_API_KEY}"}

public_no_auth = requests.get(f"{PUBLIC_URL}/v1/models", timeout=30)
assert public_no_auth.status_code == 401, (
    f"Expected public unauthenticated request to return 401, got {public_no_auth.status_code}"
)

public_models = requests.get(
    f"{PUBLIC_URL}/v1/models",
    headers=public_headers,
    timeout=60,
)
public_models.raise_for_status()
print("Public endpoint authenticated successfully.")
print(json.dumps(public_models.json(), indent=2)[:3000])

Public endpoint authenticated successfully.
{
  "object": "list",
  "data": [
    {
      "id": "hf.co/HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF:IQ4_XS",
      "object": "model",
      "created": 1793438855,
      "owned_by": "HauhauCS"
    }
  ]
}


## 14. Use from Windows and OCI

Download all three files from cell 12. Put them in one folder and rename `ollama-client.env` to `.env`. Do not commit `.env`; add it to `.gitignore`.

### Windows PowerShell

```powershell
py -3.12 -m venv .venv
.venv\Scripts\Activate.ps1
pip install -r requirements-client.txt
python gpu_client.py "Explain how a lock-free MPMC queue works in C++." --stream
```

### OCI / Debian

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements-client.txt
python gpu_client.py "Explain how an event loop works." --stream
```

### Use directly with the OpenAI Python SDK

```python
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(
    base_url=os.environ["OLLAMA_BASE_URL"],
    api_key=os.environ["OLLAMA_API_KEY"],
    timeout=1800.0,
)
response = client.chat.completions.create(
    model=os.environ["OLLAMA_MODEL"],
    messages=[{"role": "user", "content": "Hello from my client"}],
)
print(response.choices[0].message.content)
```

The tunnel URL and API token change when the Kaggle session restarts. Download a fresh `.env` from the new session and replace the old one. Kaggle sessions are temporary and subject to platform limits; this is not an always-on VPS.

## 15. Stop the worker when finished

Run this cell before ending the session. It unloads the model and stops the Cloudflare tunnel, authentication proxy, and Ollama server. Kaggle may also stop the session automatically.

In [23]:
# Graceful shutdown of the processes started by this notebook.

import os
import signal
import subprocess
import requests

try:
    subprocess.run(["ollama", "stop", MODEL_ID], capture_output=True, text=True, timeout=60)
except Exception as exc:
    print("Model unload warning:", exc)

for name, process in [
    ("cloudflared", globals().get("cloudflared_process")),
    ("auth proxy", globals().get("proxy_process")),
    ("Ollama", globals().get("ollama_process")),
]:
    if process is not None and process.poll() is None:
        try:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait(timeout=5)
        except ProcessLookupError:
            pass
        print(name, "stopped")
    else:
        print(name, "was not running")

for handle_name in ("cloudflared_log", "proxy_log", "ollama_log"):
    handle = globals().get(handle_name)
    if handle is not None and not handle.closed:
        handle.close()

print("Kaggle worker shutdown requested.")

cloudflared stopped
auth proxy stopped
Ollama stopped
Kaggle worker shutdown requested.
